# **1. Text Extraction from Case PDF**

In [3]:
!pip install pymupdf python-docx

In [4]:
import fitz  # PyMuPDF
import re
from docx import Document
import sys
import os


In [5]:
import os
from kaggle_secrets import UserSecretsClient


# Get the secret from Kaggle and set it as an environment variable
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

print("HF Secrets/API keys have been set")

HF Secrets/API keys have been set


In [6]:
def legal_cleaning(text):
    """
    Enhanced cleaning for Indian Law Reports:
    Removes margin ladders (A-H), page headers, and numbers
    while preserving structure for Legal-BERT.
    """
    # 1. Remove "Alphabet Ladders" (Standalone A-H on their own lines)
    # These appear in the margins of court reporters [cite: 1, 6, 209]
    text = re.sub(r'^\s*[A-H]\s*$', '', text, flags=re.MULTILINE)

    # 2. Remove Page Headers/Footers [cite: 109, 110]
    # Targeted patterns found in your specific document
    headers = [
        r'\[2020\]\s*1\s*S\.C\.R\.',
        r'SUPREME\s*COURT\s*REPORTS',
        r'UNION\s*OF\s*INDIA\s*v\.\s*RELIANCE\s*COMMUNICATION.*',
        r'S\.\s*RAVINDRA\s*BHAT,\s*J\.'
    ]
    for pattern in headers:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 3. Remove Standalone Page Numbers [cite: 108, 138, 189]
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 4. Standard Cleaning Logic (Preserved from your original code)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r' +', ' ', text)
    text = "".join(char for char in text if char.isprintable() or char in ['\n', '\t'])

    # Final trim to ensure no trailing blank lines from the removed ladders
    text = "\n".join([line.strip() for line in text.split('\n') if line.strip()])

    return text.strip()

In [7]:
# 1. Base directory for uploaded Kaggle Dataset
# Replace 'legal-nlp-case-files' with your actual dataset slug
DATASET_PATH = "/kaggle/input/datasets/rehanfargose/case-en-pdfs-2020" 

# 2. Hardcode the filename here
FILE_NAME = "2020_1_1_7_EN.pdf" 

# 3. Construct the full dynamic path
selected_pdf_path = os.path.join(DATASET_PATH, FILE_NAME)

# Verify if the file exists to avoid 'FileNotFound' errors later
if os.path.exists(selected_pdf_path):
    print(f"Successfully located: {selected_pdf_path}")
else:
    print(f"Error: File not found at {selected_pdf_path}")
    print("Available files:", os.listdir(DATASET_PATH))

Successfully located: /kaggle/input/datasets/rehanfargose/case-en-pdfs-2020/2020_1_1_7_EN.pdf


In [8]:
def process_document(file_path):
    # 1. Basic validation
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return None

    file_name = os.path.basename(file_path)
    print(f"Processing: {file_name}...")

    # 2. Text Extraction using PyMuPDF (fitz)
    raw_text = ""
    try:
        # Open directly from the Kaggle input path
        with fitz.open(file_path) as doc:
            raw_text = "\n".join([page.get_text("text") for page in doc])

        # 3. Pre-processing & Cleaning using your custom function
        cleaned_text = legal_cleaning(raw_text)

        # 4. Output to Kaggle's working directory for your project records
        output_dir = "/kaggle/working/Text_Extracts"
        os.makedirs(output_dir, exist_ok=True)
        
        # Save the cleaned text as a .txt file
        output_file = os.path.join(output_dir, f"{os.path.splitext(file_name)[0]}.txt")
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print(f"Success! Cleaned text saved to: {output_file}")
        return cleaned_text

    except Exception as e:
        print(f"Extraction error: {str(e)}")
        return None

In [9]:
cleaned_data = process_document(selected_pdf_path )


Processing: 2020_1_1_7_EN.pdf...
Success! Cleaned text saved to: /kaggle/working/Text_Extracts/2020_1_1_7_EN.txt


In [10]:
# Verify the output
if cleaned_data:
    print(f"Character count of cleaned text: {len(cleaned_data)}")

Character count of cleaned text: 15220


# **1.5. The Segmentation method for extracting Evidence, Facts and IPCs from cleaned Text file**

In [11]:
!pip install -U google-genai

In [12]:
import json
from google import genai
from google.genai import types
# from google.colab import userdata
import os


model_name = 'gemini-3-flash-preview'
# Get the secret from Kaggle and set it as an environment variable
user_secrets = UserSecretsClient()
# os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("GOOGLE_API_KEY")

api_key = user_secrets.get_secret("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

def extract_legal_details(legal_text):
    prompt = f"""
    You are a Legal Expert in Indian Law. Analyze the following court text and extract:
    1. FACTS: The core background of the dispute.
    2. EVIDENCE: Specific documents, testimonies, or proofs cited.
    3. IPC: Any Indian Penal Code sections or specific Acts/statutes mentioned/applicable in the case.

    Format the output as a clean JSON not a list.

    TEXT:
    {legal_text}
    """

    # Use the new models.generate_content method
    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="medium"),
            response_mime_type="application/json",
        )
    )


    try:
        data = json.loads(response.text)
        return data
    except json.JSONDecodeError:
        print("Failed to parse JSON")
        return None

In [13]:
generate_models = [m.name for m in client.models.list() 
                   if 'generateContent' in m.supported_actions]

print("Models you can actually use for generation:")
print(generate_models)

Models you can actually use for generation:
['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.0-flash', 'models/gemini-2.0-flash-001', 'models/gemini-2.0-flash-exp-image-generation', 'models/gemini-2.0-flash-lite-001', 'models/gemini-2.0-flash-lite', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-3-1b-it', 'models/gemma-3-4b-it', 'models/gemma-3-12b-it', 'models/gemma-3-27b-it', 'models/gemma-3n-e4b-it', 'models/gemma-3n-e2b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-2.5-flash-lite-preview-09-2025', 'models/gemini-3-pro-preview', 'models/gemini-3-flash-preview', 'models/gemini-3.1-pro-preview', 'models/gemini-3.1-pro-preview-customtools', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3-pro-image-preview', 'models/nano-banana-pro-preview', 'models/gemini-3.1-flash-image-preview', 'mo

In [15]:
# Get the extracted facts, evidence and basic IPCs from the text file
result = extract_legal_details(cleaned_data)
# result = {'FACTS': "The appellant, M/S. L. R. Brothers Indo Flora Ltd., a 100% Export Oriented Unit (EOU) engaged in producing cut flowers, sold goods in the Domestic Tariff Area (DTA) between 1998-99 and 2000-01 without prior approval from the Development Commissioner and without achieving the required 20% positive net foreign exchange earning as mandated by the EXIM Policy 1997-2002. The unit had imported inputs like 'Live Rose Plants' and fertilizers duty-free under Notification No. 126/94-Cus. The Revenue issued a show-cause notice demanding customs duty on the imported inputs used for DTA sales, arguing that the breach of EXIM Policy conditions nullified the exemption. The appellant contended that since cut flowers are non-excisable and grown on Indian soil, they should not attract customs duty, and further argued that the 2001 amendment to the notification should apply retrospectively to reduce their liability.", 'EVIDENCE': "Exemption Notification No. 126/94-Cus dated 03.06.1994; Amendment Notification No. 56/01-Cus dated 18.05.2001; EXIM Policy 1997-2002; Show Cause Notice dated 16.03.2001 issued by the Additional Commissioner of Central Excise; Letter dated 06.02.2001 from the appellant seeking ex-post facto approval; Scrutiny of import records for capital goods, 'Live Rose Plants', fertilizers, and planting materials; Export records showing flowers worth Rs. 91.92 lakhs against capital goods imports of Rs. 12.11 crores; CBEC Circular No. 31/2001-Cus dated 24.05.2001; Appendix 42 of the Handbook of Procedure.", 'IPC': 'Customs Act, 1962 (Section 12, Section 25, Section 28, Section 28AB, Section 114A); Central Excise Act, 1944 (Section 3); Customs Tariff Act, 1975; Central Excise Tariff Act, 1985; EXIM Policy 1997-2002.'}
print(result)

{'FACTS': "The dispute centers on the refund of excess money encashed by the Union of India from bank guarantees provided by Reliance Communications Ltd (RCL) and Reliance Telecom Ltd (RTL). The respondents were successful bidders in spectrum auctions under NIA 2013 and NIA 2015 but faced financial distress, leading to defaults in deferred spectrum charges. The Union encashed bank guarantees totaling Rs. 908.91 crores against an admitted liability of Rs. 774.25 crores. Although the respondents furnished fresh bank guarantees for subsequent installments, the Union refused to refund the surplus of approximately Rs. 134.66 crores, citing subsequent defaults and potential future liabilities. The TDSAT ordered a partial refund of Rs. 104.34 crores after adjustments, a decision which the Supreme Court upheld, dismissing the Union's appeal.", 'EVIDENCE': ['Notice Inviting Applications (NIA) 2013 and NIA 2015 for spectrum auctions', 'Bank Guarantees totaling Rs. 908.91 crores and subsequent gu

In [16]:
print(f"The processed facts using {model_name} are:")
case_facts = result.get('FACTS')
# case_facts = "M/S. L.R. Brothers Indo Flora Ltd., a 100% Export Oriented Unit (EOU) producing cut flowers, imported capital goods and raw materials (live rose plants, fertilizers) duty-free under Notification No. 126/94-Cus. Between 1998-99 and 2000-01, the appellant made sales in the Domestic Tariff Area (DTA) totaling Rs. 38,40,537 without obtaining the mandatory prior approval from the Development Commissioner and without achieving the required 20% positive net foreign exchange earning as stipulated by the EXIM Policy 1997-2002. The Revenue issued a show cause notice seeking customs duty on the imported inputs used for the DTA sales of non-excisable goods (cut flowers), calculating the duty at a rate equivalent to the customs duty leviable on the finished articles if imported."
print(case_facts)

The processed facts using gemini-3-flash-preview are:
The dispute centers on the refund of excess money encashed by the Union of India from bank guarantees provided by Reliance Communications Ltd (RCL) and Reliance Telecom Ltd (RTL). The respondents were successful bidders in spectrum auctions under NIA 2013 and NIA 2015 but faced financial distress, leading to defaults in deferred spectrum charges. The Union encashed bank guarantees totaling Rs. 908.91 crores against an admitted liability of Rs. 774.25 crores. Although the respondents furnished fresh bank guarantees for subsequent installments, the Union refused to refund the surplus of approximately Rs. 134.66 crores, citing subsequent defaults and potential future liabilities. The TDSAT ordered a partial refund of Rs. 104.34 crores after adjustments, a decision which the Supreme Court upheld, dismissing the Union's appeal.


In [17]:
print(f"The processed evidence using {model_name} are:")
case_evidence = result.get('EVIDENCE')
# case_evidence = "Notification No. 126/94-Cus (dated 03.06.1994); Amendment Notification No. 56/01-Cus (dated 18.05.2001); EXIM Policy 1997-2002 (Paragraphs 9.5, 9.9, 9.29); Letter of approval No. 119(1994)EOB/34/94 (dated 04.05.1994); Appellant's letter seeking ex-post facto approval (dated 06.02.2001); Show cause notice issued by Additional Commissioner, Central Excise, Meerut-I (dated 16.03.2001); CBEC Circular No. 31/2001-Cus (dated 24.05.2001); Import documentation for Live Rose Plants, fertilizers, and greenhouse equipment."

print(case_evidence)

The processed evidence using gemini-3-flash-preview are:
['Notice Inviting Applications (NIA) 2013 and NIA 2015 for spectrum auctions', 'Bank Guarantees totaling Rs. 908.91 crores and subsequent guarantees of Rs. 774.25 crores', 'Scheme for Amalgamation under the Companies Act, 1956 (Sistema merged with RCL)', 'Modified payment terms letters dated 19.03.2018 issued by the Union', 'Strategic Debt Restructuring (SDR) framework documents and RBI guidelines', 'Telecom Dispute Settlement & Appellate Tribunal (TDSAT) order dated 21.12.2018', 'NCLAT (National Company Law Appellate Tribunal) stay orders regarding subsequent defaults', 'Clauses 13.1 and 13.2 of the license agreements']


In [18]:
print(f"The processed IPCs using {model_name} are:")
case_ipc = result.get('IPC')
# case_ipc = "Customs Act, 1962 (Sections 12, 25, 28, 28AB, 114A); Central Excise Act, 1944 (Section 3); Customs Tariff Act, 1975; Central Excise Tariff Act, 1985; EXIM Policy 1997-2002."
print(case_ipc)

The processed IPCs using gemini-3-flash-preview are:
['Companies Act, 1956', 'Insolvency and Bankruptcy Code (IBC)', 'Clause 4.5b(ix) of NIA 2013', 'Clause 4.5b(x) of NIA 2015']


# **Load the Llama 3 model**

In [19]:
# !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [20]:
# !pip install --no-deps "xformers>=0.0.35" "trl>=0.27.0" peft accelerate bitsandbytes

In [21]:
!pip install -U peft transformers accelerate

In [22]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

In [23]:
base_model_id = "unsloth/meta-llama-3.1-8b-instruct"

In [24]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

In [25]:
# 2. Load the base model in FP16
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.float16,  # This replaces the 4-bit quantization
    device_map="auto",  
    trust_remote_code=True
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [26]:
model = PeftModel.from_pretrained(base_model, "Raazi29/Nyaya-Llama-3.1-8B-Indian-Legal")


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/671M [00:00<?, ?B/s]

In [27]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

# **2.5. Scrutinize the Evidence based on Facts and Arguments**

In [28]:
# scrutiny_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
# You are an expert Legal Auditor specializing in evidentiary scrutiny. 
# Your task is to perform a strict comparative analysis between Alleged FACTS and provided EVIDENCE.

# AUDIT CRITERIA:
# 1. QUANTITATIVE: Identify mismatches in currency, figures, percentages, or units[cite: 169, 184].
# 2. TEMPORAL: Identify discrepancies in dates, periods of limitation, or sequence of events[cite: 158, 465].
# 3. AUTHORITATIVE: Identify actions taken without mandatory prior approval or jurisdiction[cite: 158, 213].
# 4. STATUTORY: Identify claims of exemption where evidence shows non-compliance with notification conditions[cite: 41, 83].

# OUTPUT FORMAT:
# - List each finding as a "CONTRADICTION" or "GAP".
# - You must reference specific values, dates, and document titles from the text.
# - Be concise. Do not explain the law unless it highlights a discrepancy.
# <|eot_id|><|start_header_id|>user<|end_header_id|>

# ### CASE MATERIALS
# FACTS:
# {case_facts}

# ### EVIDENCE:
# {case_evidence}

# ### AUDIT RESULTS:
# 1. """


scrutiny_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Senior Legal Auditor. Your task is to perform a granular "Requirement vs. Reality" audit.
For every finding, you MUST include:
- THE STATUTORY REQUIREMENT: (e.g., EXIM Policy para 9.5 or Notification 126/94-Cus).
- THE FACTUAL REALITY: (Exact INR values, Dates, and Document IDs).
- THE QUANTIFIABLE DISCREPANCY: (The specific math showing the violation).

Be verbose and forensic. Do not summarize.
<|eot_id|><|start_header_id|>user<|end_header_id|>

### CASE MATERIALS
FACTS:
{case_facts}

EVIDENCE:
{case_evidence}

### DETAILED AUDIT ANALYSIS:
1. QUANTITATIVE SHORTFALL:"""

In [29]:
# Tokenize prompt
inputs = tokenizer(scrutiny_prompt, return_tensors="pt").to("cuda")

In [30]:
print(f"Input tensor shape: {inputs.input_ids.shape}")

Input tensor shape: torch.Size([1, 490])


In [31]:
with torch.inference_mode():
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        min_new_tokens=400,
        max_new_tokens=2048,
        do_sample=False,            # Greedy decoding for precision
        # no_repeat_ngram_size=5,     
        repetition_penalty=1.1, 
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [32]:
generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]

# 3. Decode and print
scrutiny_result = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("### SCRUTINY REPORT ###")
print(scrutiny_result)

### SCRUTINY REPORT ###
 
   - THE STATUTORY REQUIREMENT: Clause 13.1 of License Agreement.
   - THE FACTUAL REALITY: Modified Payment Terms Letter No. 01/2018 dated 19.03.2018.
   - THE QUANTIFIABLE DISCREPANCY: Excess amount encashed = Rs. 134.66 Crore.

2. QUALITATIVE NON-COMPLIANCE: 
   - THE STATUTORY REQUIREMENT: Clause 13.2 Of License Agreement.
   - THE FACTUAL REALITY: Scheme For Amalgamation Under The Companies Act, 1956.
   - THE QUANTIFIABLE DISCREPANCY: Non-refund Of Surplus Amount Despite Subsequent Defaults And Potential Future Liabilities.

3. LEGAL JUDGMENT: 
   - THE STATUTORY REQUIREMENT: Judgment In <b>Union</b> Of <b>India</b> vs <b>Reliance</b> <b>Communications</b> <b>Ltd.</b> And Ors. on 30 April, 2019.
   - THE FACTUAL REALITY: Tdsat Order Dated 21.12.2018 And Nclat Stay Orders Regarding Subsequent Defaults.
   - THE QUANTIFIABLE DISCREPANCY: Partial Refund Of Rs. 104.34 Crores As Per Tdsat Order. 

4. REASONED JUDGMENT: 
   - THE STATUTORY REQUIREMENT: Clauses

# **4. Precedent Retrieval**

In [33]:
precedent_prompt = f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Senior Legal Researcher. Your task is to identify relevant Indian Case Law precedents (Supreme Court or High Court) that align with the following case details. 
Focus on cases involving similar IPC sections.
Be specific with case names and years if possible.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
### CURRENT CASE FACTS:
{case_facts}

### APPLICABLE IPC/STATUTES:
{case_ipc}


### TASK:
1. Identify 2-3 landmark or relevant Indian precedents.
2. Explain the 'Ratio Decidendi' (legal reasoning) of those cases.
3. State how they apply to the current disputed evidence.

### RELEVANT PRECEDENTS:<|assistant|><|end_header_id|>
"""

In [34]:
# Tokenize prompt
inputs = tokenizer(precedent_prompt, return_tensors="pt").to("cuda")

In [35]:
print(f"Input tensor shape: {inputs.input_ids.shape}")

Input tensor shape: torch.Size([1, 365])


In [36]:
with torch.inference_mode():
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        min_new_tokens=100,
        max_new_tokens=1024,
        do_sample=False,            # Greedy decoding for precision
        # no_repeat_ngram_size=5,     
        repetition_penalty=1.1, 
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [37]:
generated_tokens1 = outputs[0][inputs.input_ids.shape[-1]:]

# 3. Decode and print
precedent_result = tokenizer.decode(generated_tokens1, skip_special_tokens=True)

print("### PRECEDENT REPORT ###")
print(precedent_result)

### PRECEDENT REPORT ###
<b>National Insurance Co. Ltd vs Pranayraj And Ors.</b>, <b>(2007) 9 Scc 641</b>

<b>Rajesh Bhai Patel & Anr vs State Of Gujarat & Ors</b>, <b>2018(1) Guj.Law Times 1</b>

<b>State Of U.P. vs M/S. Rameshwar Prasad Gupta & Sons</b>, <b>1960(1) Slr 1</b>


# **5. Verdict/Sentence Prediction**

In [58]:
# verdict_prompt = f"""
# <|begin_of_text|><|start_header_id|>system<|end_header_id|>
# You are an experienced Indian High Court Judge. Your task is to provide a reasoned verdict based on the facts, the applicable IPC sections, and the scrutinized evidence analysis. 

# Follow these steps in your reasoning:
# 1. STATUTORY ALIGNMENT: Match the facts against the essential ingredients of the cited IPC/Statutes sections.
# 2. EVIDENTIARY WEIGHT: Assess if the evidence scrutiny results (contradictions/gaps) create "reasonable doubt."
# 3. VERDICT: Declare the defendant Guilty or Not Guilty.
# 4. PRECEDENTS: Consider the precedents found and adjust them accordingly for the current case.
# 5. SENTENCING: If guilty, suggest a sentence based on standard Indian judicial precedents for these offenses.

# Be formal, objective, and authoritative.<|eot_id|>

# <|start_header_id|>user<|end_header_id|>
# ### ESTABLISHED FACTS:
# {case_facts}

# ### APPLICABLE IPC/STATUTES:
# {case_ipc}

# ### EVIDENCE SCRUTINY ANALYSIS:
# {scrutiny_result}


# ### JUDICIAL REASONING & VERDICT:<|assistant|><|end_header_id|>
# """


verdict_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Justice of the Supreme Court of India. Decide the verdict for the given case below. 
Do not reference outside cases unless they are in the context. 
Your verdict must be based ONLY on the following materials.

### CASE MATERIALS
FACTS: {case_facts}
STATUTES: {case_ipc}
SCRUTINY: {scrutiny_result}

### JUDICIAL PRONOUNCEMENT
1. ON STATUTORY BREACH:"""

In [59]:
# Tokenize prompt
inputs = tokenizer(verdict_prompt, return_tensors="pt").to("cuda")

In [60]:
print(f"Input tensor shape: {inputs.input_ids.shape}")

Input tensor shape: torch.Size([1, 802])


In [61]:
with torch.inference_mode():
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        min_new_tokens=100,
        max_new_tokens=800,
        do_sample=True,            # Greedy decoding for precision
        no_repeat_ngram_size=5,     
        repetition_penalty=1.1, 
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

Both `max_new_tokens` (=800) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [62]:
generated_tokens2 = outputs[0][inputs.input_ids.shape[-1]:]

# 3. Decode and print
verdict_result = tokenizer.decode(generated_tokens2, skip_special_tokens=True)

print("### VERDICT REPORT ###")
print(verdict_result)

### VERDICT REPORT ###
 
The Union Of India Has Failed To Comply With The Statutory Requirement Under Clause 13.3 Of The License Agreement Which Mandates A Refund Of Excess Amount If It Is Not Utilized Within 45 Days. Further, The Union Fails To Comply With Clause 13.4 Of The License Agreement That Requires It To Refund The Excess Amount Immediately Upon Request By The Licensee. Therefore, The Union's Failure To Comply With These Provisions Renders Its Actions Unjustified.


# **6. Summarization**

In [63]:
summary_prompt = f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Legal Journalist. Summarize the following legal case in approximately 150 words. 
Use simple, non-technical language that a non-lawyer can understand. 
Structure the summary as:
1. The Core Dispute (What happened?)
2. Key Findings (What did the analysis reveal?)
3. Final Verdict & Reasoning (What was decided and why?)
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
### CASE DATA:
- **FACTS**: {case_facts}
- **SCRUTINIZED EVIDENCE**: {scrutiny_result}
- **RELEVANT PRECEDENTS**: {precedent_result}
- **PREDICTED VERDICT**: {verdict_result}

### SIMPLE SUMMARY:<|assistant|><|end_header_id|>
"""

In [64]:
# Tokenize prompt
inputs = tokenizer(summary_prompt, return_tensors="pt").to("cuda")

In [65]:
print(f"Input tensor shape: {inputs.input_ids.shape}")

Input tensor shape: torch.Size([1, 1021])


In [66]:
with torch.inference_mode():
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        min_new_tokens=100,
        max_new_tokens=800,
        do_sample=True,            # Greedy decoding for precision
        no_repeat_ngram_size=5,     
        repetition_penalty=1.1, 
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

Both `max_new_tokens` (=800) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [67]:
generated_tokens3 = outputs[0][inputs.input_ids.shape[-1]:]

# 3. Decode and print
summary_result = tokenizer.decode(generated_tokens3, skip_special_tokens=True)

print("### SUMMARY REPORT ###")
print(summary_result)

### SUMMARY REPORT ###
This legal judgment is about a dispute between the Union Of India and Reliance Communications Ltd. And Reliance Telecom Ltd. over the refund of an excess amount encashed by the former from bank guarantees provided. The Union Of India had encashed bank guarantees worth Rs. 908 crore, while it only needed Rs. 774 crore. After further defaults, it refused to return the remaining Rs. 134 crore. The Tribunal For The Disputes Settlement And Appellate Tribunal For Major Ports held that the Union must refund this excess amount. This reasoning was further supported by the Supreme Court.
